# Agricultural Futures Final Project

## Corn Futures + MODIS NDVI Regime + Course Strategy Extensions

**Project question**

Can a low-frequency vegetation signal from NASA MODIS improve a systematic agricultural-futures process when layered on top of multiple course-style trading strategies?

**Current scope**

- Primary market: Corn futures
- Optional second market: Wheat futures, if the local cache exists
- NASA regime: MODIS `MOD13A2` NDVI
- Course strategies ported into this notebook:
  - trend following (`10/30`, `30/100`, `80/160`)
  - counter-trend bounce
  - volatility-regime rule
  - optional corn-wheat pairs/spread strategy
  - simple weighted strategy combination

The notebook is designed to run from local CSV caches by default. Separate builder notebooks handle any live Yahoo refresh work.


In [ ]:
from pathlib import Path
import os
import re
from datetime import datetime
from itertools import product
import warnings

os.environ.setdefault('MPLCONFIGDIR', '/Users/jlaw/projects/stern/systematic-investing/.mplconfig')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings('ignore', category=FutureWarning)

PROJECT_ROOT = Path('/Users/jlaw/projects/stern/systematic-investing')
DATA_ROOT = PROJECT_ROOT / 'data' / 'ag_futures'
RAW_FUTURES_DIR = DATA_ROOT / 'raw' / 'futures'
RAW_NASA_DIR = DATA_ROOT / 'raw' / 'nasa' / 'mod13a2'
PROCESSED_DIR = DATA_ROOT / 'processed'

EARTHDATA_ROOT_CANDIDATES = [
    Path('/Users/jlaw/projects/earthdata_downloads'),
    RAW_NASA_DIR,
]

ROLL_MONTHS = [3, 5, 7, 9, 12]
MA_PAIRS = [(10, 30), (30, 100), (80, 160)]
PRIMARY_PAIR = (30, 100)
MIN_NDVI_FILES = 12

CORN_CACHE_PATH = PROCESSED_DIR / 'corn_continuous_back_adjusted.csv'
WHEAT_CACHE_PATH = PROCESSED_DIR / 'wheat_continuous_back_adjusted.csv'
NDVI_LEVEL_CACHE_PATH = PROCESSED_DIR / 'mod13a2_ndvi_level_series.csv'
NDVI_DAILY_CACHE_PATH = PROCESSED_DIR / 'mod13a2_ndvi_daily_series.csv'
PAIRS_CACHE_PATH = PROCESSED_DIR / 'corn_wheat_pairs_panel.csv'

REFRESH_CORN_FROM_YAHOO = False
REFRESH_NDVI_FROM_RAW = False
RUN_WHEAT_PAIRS_ANALYSIS = True
COMBINATION_GRID_STEP = 0.1
MAX_STRATEGIES_IN_COMBO = 4

for path in [RAW_FUTURES_DIR, RAW_NASA_DIR, PROCESSED_DIR]:
    path.mkdir(parents=True, exist_ok=True)


def pick_first_existing_path(candidates):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def parse_modis_vi_date(path):
    match = re.search(r'\.A(\d{4})(\d{3})\.', path.name)
    if not match:
        return pd.NaT
    year = int(match.group(1))
    doy = int(match.group(2))
    return pd.Timestamp(datetime.strptime(f'{year}-{doy:03d}', '%Y-%j'))


def build_ndvi_inventory(root):
    if root is None:
        return pd.DataFrame(columns=['path', 'date', 'year', 'doy', 'tile'])
    rows = []
    for path in sorted(root.rglob('MOD13A2*.hdf')):
        date = parse_modis_vi_date(path)
        tile_match = re.search(r'\.(h\d{2}v\d{2})\.', path.name)
        rows.append({
            'path': str(path),
            'date': date,
            'year': date.year if pd.notna(date) else np.nan,
            'doy': date.dayofyear if pd.notna(date) else np.nan,
            'tile': tile_match.group(1) if tile_match else None,
        })
    if not rows:
        return pd.DataFrame(columns=['path', 'date', 'year', 'doy', 'tile'])
    inventory = (
        pd.DataFrame(rows)
        .drop_duplicates(subset=['date', 'tile', 'path'])
        .sort_values(['date', 'tile', 'path'])
        .reset_index(drop=True)
    )
    return inventory


def ndvi_inventory_diagnostics(inventory):
    if inventory.empty:
        return {
            'file_count': 0,
            'unique_dates': 0,
            'tiles': [],
            'median_gap_days': np.nan,
            'max_gap_days': np.nan,
        }
    unique_dates = pd.Series(sorted(pd.to_datetime(inventory['date'].dropna().unique())))
    if unique_dates.shape[0] > 1:
        gaps = unique_dates.diff().dropna().dt.days
        median_gap_days = float(gaps.median())
        max_gap_days = int(gaps.max())
    else:
        median_gap_days = np.nan
        max_gap_days = np.nan
    return {
        'file_count': int(len(inventory)),
        'unique_dates': int(unique_dates.shape[0]),
        'tiles': sorted(inventory['tile'].dropna().unique().tolist()),
        'median_gap_days': median_gap_days,
        'max_gap_days': max_gap_days,
    }


def annualized_sharpe(returns, periods=252):
    series = pd.Series(returns).dropna()
    if series.empty or series.std() == 0:
        return np.nan
    return np.sqrt(periods) * series.mean() / series.std()


def max_drawdown(returns):
    series = pd.Series(returns).fillna(0.0)
    wealth = (1.0 + series).cumprod()
    drawdown = wealth / wealth.cummax() - 1.0
    return drawdown.min()


def summarize_strategy(name, returns, signal=None):
    series = pd.Series(returns).dropna()
    if series.empty:
        cum_return = np.nan
        ann_vol = np.nan
        mdd = np.nan
    else:
        cum_return = (1.0 + series).prod() - 1.0
        ann_vol = series.std() * np.sqrt(252)
        mdd = max_drawdown(series)
    summary = {
        'strategy': name,
        'obs': int(series.shape[0]),
        'cum_return': cum_return,
        'ann_sharpe': annualized_sharpe(series),
        'ann_vol': ann_vol,
        'max_drawdown': mdd,
    }
    if signal is not None:
        sig = pd.Series(signal).dropna()
        summary['active_share'] = sig.abs().mean() if not sig.empty else np.nan
    return summary


def normalize_yfinance_frame(frame):
    data = frame.copy()
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)
    keep = [col for col in ['Open', 'High', 'Low', 'Close', 'Volume'] if col in data.columns]
    return data[keep].dropna().copy()


def back_adjust_continuous_futures(frame):
    data = frame.copy()
    data['month'] = data.index.month
    data['is_roll_switch'] = data['month'].isin(ROLL_MONTHS) & (data['month'] != data['month'].shift(1))
    data['daily_change'] = data['Close'].diff()
    data['typical_change'] = (data['daily_change'].shift(1) + data['daily_change'].shift(-1)) / 2.0
    data['roll_gap'] = 0.0
    data.loc[data['is_roll_switch'], 'roll_gap'] = (
        data.loc[data['is_roll_switch'], 'daily_change'] - data.loc[data['is_roll_switch'], 'typical_change']
    )
    data['roll_gap'] = data['roll_gap'].fillna(0.0)
    data['cum_adjustment'] = data['roll_gap'].iloc[::-1].cumsum().iloc[::-1].shift(-1).fillna(0.0)
    data['Adj_Close_Futures'] = data['Close'] + data['cum_adjustment']
    data['futures_ret'] = data['Adj_Close_Futures'].pct_change()
    return data


def add_ma_signals(frame, pairs):
    data = frame.copy()
    daily_pct = data['Adj_Close_Futures'].pct_change()
    for short_window, long_window in pairs:
        short_col = f'ma{short_window}'
        long_col = f'ma{long_window}'
        signal_col = f'signal_{short_window}_{long_window}'
        strat_col = f'strategy_ret_{short_window}_{long_window}'
        data[short_col] = daily_pct.rolling(short_window).mean()
        data[long_col] = daily_pct.rolling(long_window).mean()
        signal = pd.Series(
            np.where(data[short_col] > data[long_col], 1.0, -1.0),
            index=data.index,
            dtype='float64',
        )
        signal[data[short_col].isna() | data[long_col].isna()] = np.nan
        data[signal_col] = signal
        data[strat_col] = data[signal_col].shift(1) * data['futures_ret']
    return data


def load_or_download_futures_cache(symbol, cache_path, start_date, refresh=False):
    if cache_path.exists() and not refresh:
        data = pd.read_csv(cache_path, parse_dates=['Date']).set_index('Date').sort_index()
        print(f'Loaded cached series from {cache_path}')
        return data
    frame = yf.download(symbol, start=start_date, auto_adjust=False, progress=False, actions=False)
    if frame.empty:
        raise RuntimeError(
            f'yfinance returned no data for {symbol}. If you want a CSV-only workflow, keep refresh=False and populate the cache first.'
        )
    frame = normalize_yfinance_frame(frame)
    frame = back_adjust_continuous_futures(frame)
    frame = add_ma_signals(frame, MA_PAIRS)
    frame.to_csv(cache_path, index_label='Date')
    print(f'Saved refreshed series to {cache_path}')
    return frame


def load_local_futures_cache(cache_path, label):
    if not cache_path.exists():
        print(f'No local {label} cache found at {cache_path}.')
        return None
    frame = pd.read_csv(cache_path, parse_dates=['Date']).set_index('Date').sort_index()
    print(f'Loaded local {label} cache from {cache_path}')
    return frame


def require_pyhdf():
    try:
        from pyhdf.SD import SD, SDC
    except ImportError as exc:
        raise ImportError(
            'pyhdf is required to read MOD13A2 HDF4 files. Install it in the notebook environment or re-download the NDVI subset through AppEEARS in NetCDF format.'
        ) from exc
    return SD, SDC


def read_mod13a2_ndvi_mean(path):
    SD, SDC = require_pyhdf()
    hdf = SD(str(path), SDC.READ)
    sds = hdf.select('1 km 16 days NDVI')
    arr = sds.get().astype('float64')
    attrs = sds.attributes()
    fill_value = attrs.get('_FillValue', attrs.get('fillvalue', -3000))
    arr[arr == fill_value] = np.nan
    valid_range = attrs.get('valid_range')
    if valid_range is not None and len(valid_range) == 2:
        arr[(arr < valid_range[0]) | (arr > valid_range[1])] = np.nan
    scale_factor = attrs.get('scale_factor', 0.0001)
    if scale_factor in [None, 0]:
        scale_factor = 0.0001
    if scale_factor > 1:
        arr = arr / scale_factor
    else:
        arr = arr * scale_factor
    return float(np.nanmean(arr))


def load_ndvi_series(inventory):
    if inventory.empty:
        empty = pd.DataFrame(columns=['ndvi', 'ndvi_roll_mean', 'ndvi_roll_std', 'ndvi_z', 'ndvi_regime'])
        return empty, empty
    if NDVI_LEVEL_CACHE_PATH.exists() and not REFRESH_NDVI_FROM_RAW:
        ndvi = pd.read_csv(NDVI_LEVEL_CACHE_PATH, parse_dates=['date']).set_index('date').sort_index()
        print(f'Loaded cached NDVI means from {NDVI_LEVEL_CACHE_PATH}')
    else:
        records = []
        for idx, row in enumerate(inventory.itertuples(index=False), start=1):
            ndvi_value = read_mod13a2_ndvi_mean(Path(row.path))
            records.append({'date': row.date, 'tile': row.tile, 'ndvi': ndvi_value})
            if idx % 50 == 0 or idx == len(inventory):
                print(f'Processed {idx}/{len(inventory)} MOD13A2 files')
        ndvi = (
            pd.DataFrame(records)
            .dropna()
            .groupby('date', as_index=False)['ndvi']
            .mean()
            .sort_values('date')
            .set_index('date')
        )
        ndvi.to_csv(NDVI_LEVEL_CACHE_PATH, index_label='date')
    ndvi['ndvi_roll_mean'] = ndvi['ndvi'].rolling(12, min_periods=6).mean()
    ndvi['ndvi_roll_std'] = ndvi['ndvi'].rolling(12, min_periods=6).std()
    ndvi['ndvi_z'] = (ndvi['ndvi'] - ndvi['ndvi_roll_mean']) / ndvi['ndvi_roll_std']
    ndvi['ndvi_regime'] = np.where(
        ndvi['ndvi_z'] < -1.0,
        1.0,
        np.where(ndvi['ndvi_z'] > 1.0, -1.0, 0.0),
    )
    full_index = pd.date_range(ndvi.index.min(), ndvi.index.max(), freq='D')
    ndvi_daily = ndvi.reindex(full_index).ffill()
    ndvi_daily.index.name = 'Date'
    ndvi_daily.to_csv(NDVI_DAILY_CACHE_PATH, index_label='Date')
    return ndvi, ndvi_daily


def apply_ndvi_overlay(frame, primary_pair):
    data = frame.copy()
    signal_col = f'signal_{primary_pair[0]}_{primary_pair[1]}'
    data['overlay_signal'] = np.where(
        data[signal_col].isna(),
        np.nan,
        np.where(
            data['ndvi_regime'] == data[signal_col],
            data[signal_col],
            np.where(data['ndvi_regime'] == 0.0, 0.5 * data[signal_col], 0.0),
        ),
    )
    data['overlay_ret'] = data['overlay_signal'].shift(1) * data['futures_ret']
    return data


def run_countertrend_strategy(frame, p=2.2, avg_range_window=20, prior_high_window=20):
    data = frame[['Open', 'High', 'Low', 'Close', 'Adj_Close_Futures', 'futures_ret']].copy()
    data['DayRange'] = data['High'] - data['Low']
    data['Av20R'] = data['DayRange'].shift(1).rolling(avg_range_window).mean()
    data['PrvHiWRoll'] = data['High'].shift(1).rolling(prior_high_window).max()
    data['HitLevel'] = data['PrvHiWRoll'] - (p * data['Av20R'])
    data['Hit?'] = data['Low'] < data['HitLevel']
    data['HitAt'] = np.where(data['Open'] < data['HitLevel'], data['Open'], data['HitLevel'])
    data['HitAt'] = np.where(data['Hit?'], data['HitAt'], np.nan)
    data['ExitPr'] = data['Close']
    data['strategy_return'] = np.where(data['Hit?'], (data['ExitPr'] - data['HitAt']) / data['HitAt'], 0.0)
    data['signal'] = np.where(data['Hit?'], 1.0, 0.0)
    data['cumulative_return'] = (1.0 + pd.Series(data['strategy_return']).fillna(0.0)).cumprod() - 1.0
    return data


def run_vol_regime_strategy(frame, vol_window=20, z_window=250, neutral_low=0.0, neutral_high=0.5):
    data = frame[['futures_ret']].copy()
    data['vol20'] = data['futures_ret'].rolling(vol_window).std()
    mean_ref = data['vol20'].shift(1).rolling(z_window).mean()
    std_ref = data['vol20'].shift(1).rolling(z_window).std()
    data['zvol20'] = (data['vol20'] - mean_ref) / std_ref
    data['signal'] = np.where((data['zvol20'] < neutral_low) | (data['zvol20'] > neutral_high), 1.0, -1.0)
    data.loc[data['zvol20'].isna(), 'signal'] = np.nan
    data['strategy_return'] = data['signal'].shift(1) * data['futures_ret']
    data['cumulative_return'] = (1.0 + data['strategy_return'].fillna(0.0)).cumprod() - 1.0
    return data


def dickey_fuller_diagnostics(spread):
    series = pd.Series(spread).dropna()
    test = pd.DataFrame({'spread': series})
    test['delta_spread'] = test['spread'].diff()
    test['lag_spread'] = test['spread'].shift(1)
    test = test.dropna()
    model = sm.OLS(test['delta_spread'], sm.add_constant(test['lag_spread'])).fit()
    adf_stat, pvalue, usedlag, nobs, critical_values, _ = adfuller(series, regression='c', autolag='AIC')
    return {
        'df_tstat_regression': float(model.tvalues['lag_spread']),
        'adf_stat': float(adf_stat),
        'pvalue': float(pvalue),
        'critical_10pct': critical_values['10%'],
        'critical_5pct': critical_values['5%'],
        'critical_1pct': critical_values['1%'],
        'observations': int(nobs),
    }


def build_pairs_panel(corn_frame, wheat_frame, horizons=(5, 10, 20)):
    pair = pd.DataFrame({
        'corn': corn_frame['Adj_Close_Futures'],
        'wheat': wheat_frame['Adj_Close_Futures'],
    }).dropna().copy()
    pair['spread_price'] = pair['corn'] - pair['wheat']
    pair['log_spread'] = np.log(pair['corn']) - np.log(pair['wheat'])
    pair['ret1CORN'] = pair['corn'].pct_change()
    pair['ret1WHEAT'] = pair['wheat'].pct_change()
    for horizon in horizons:
        for label in ['CORN', 'WHEAT']:
            price_col = 'corn' if label == 'CORN' else 'wheat'
            pair[f'{label}_ret{horizon}'] = pair[price_col].pct_change(horizon)
            pair[f'z{label}{horizon}'] = (
                (pair[f'{label}_ret{horizon}'] - pair[f'{label}_ret{horizon}'].shift(1).rolling(60).mean())
                / pair[f'{label}_ret{horizon}'].shift(1).rolling(60).std()
            )
        pair[f'zdiff{horizon}'] = pair[f'zCORN{horizon}'] - pair[f'zWHEAT{horizon}']
    pair['vol20CORN'] = pair['ret1CORN'].rolling(20).std()
    pair['vol20WHEAT'] = pair['ret1WHEAT'].rolling(20).std()
    pair['invvolCORN'] = 1.0 / pair['vol20CORN']
    pair['invvolWHEAT'] = 1.0 / pair['vol20WHEAT']
    pair['sum_invvol'] = pair['invvolCORN'] + pair['invvolWHEAT']
    pair['wCORN'] = pair['invvolCORN'] / pair['sum_invvol']
    pair['wWHEAT'] = pair['invvolWHEAT'] / pair['sum_invvol']
    pair['future_spread_ret'] = pair['wCORN'] * pair['ret1CORN'].shift(-1) - pair['wWHEAT'] * pair['ret1WHEAT'].shift(-1)
    return pair


def run_pairs_position_strategy(spread_col, fret1, long_entry=-1.0, short_entry=1.0, long_cap=1.0, short_cap=-1.0, max_holding=5):
    age = 0
    ages = []
    positions = []
    current_position = 0
    for spread in spread_col:
        if current_position == 1 and (spread >= long_cap or age >= max_holding):
            current_position = 0
            age = 0
        elif current_position == -1 and (spread <= short_cap or age >= max_holding):
            current_position = 0
            age = 0
        if current_position == 0:
            if spread <= long_entry:
                current_position = 1
                age = 1
            elif spread >= short_entry:
                current_position = -1
                age = 1
        else:
            age += 1
        ages.append(age)
        positions.append(current_position)
    results = pd.DataFrame({'position': positions, 'age': ages}, index=pd.Index(spread_col.index))
    results['strategy_return'] = results['position'] * fret1
    results['cumulative_return'] = (1.0 + results['strategy_return'].fillna(0.0)).cumprod() - 1.0
    return results


def build_weight_grid(n_assets, step=0.1):
    units = int(round(1.0 / step))
    combos = []
    for values in product(range(units + 1), repeat=n_assets):
        if sum(values) == units:
            combos.append(np.array(values, dtype=float) / units)
    return combos


def optimize_strategy_combo(returns_frame, top_n=4, step=0.1):
    summaries = []
    for col in returns_frame.columns:
        summaries.append(summarize_strategy(col, returns_frame[col]))
    summary_df = pd.DataFrame(summaries).set_index('strategy')
    ranked = summary_df.sort_values('ann_sharpe', ascending=False)
    candidate_names = ranked[ranked['ann_sharpe'] > 0].head(top_n).index.tolist()
    if len(candidate_names) < 2:
        return summary_df, None, None, candidate_names, None
    combo_frame = returns_frame[candidate_names].dropna()
    if combo_frame.empty:
        return summary_df, None, None, candidate_names, None
    best = None
    best_returns = None
    for weights in build_weight_grid(len(candidate_names), step=step):
        combined = combo_frame.mul(weights, axis=1).sum(axis=1)
        sharpe = annualized_sharpe(combined)
        drawdown = max_drawdown(combined)
        if np.isnan(sharpe):
            continue
        if best is None or sharpe > best['ann_sharpe'] or (np.isclose(sharpe, best['ann_sharpe']) and drawdown > best['max_drawdown']):
            best = {
                'weights': dict(zip(candidate_names, weights)),
                'ann_sharpe': sharpe,
                'cum_return': (1.0 + combined).prod() - 1.0,
                'ann_vol': combined.std() * np.sqrt(252),
                'max_drawdown': drawdown,
                'obs': int(combined.shape[0]),
            }
            best_returns = combined
    equal_weight_returns = combo_frame.mean(axis=1)
    return summary_df, best, best_returns, candidate_names, equal_weight_returns


In [ ]:
earthdata_root = pick_first_existing_path(EARTHDATA_ROOT_CANDIDATES)
ndvi_inventory = build_ndvi_inventory(earthdata_root)
ndvi_diag = ndvi_inventory_diagnostics(ndvi_inventory)

print(f'Project root: {PROJECT_ROOT}')
print(f'Earthdata root used for MOD13A2 search: {earthdata_root}')
print(f'NDVI files found: {ndvi_diag["file_count"]}')
if not ndvi_inventory.empty:
    print(f'NDVI coverage from filename dates: {ndvi_inventory["date"].min().date()} to {ndvi_inventory["date"].max().date()}')
    print(f'Unique composite dates: {ndvi_diag["unique_dates"]}')
    print(f'Tiles detected: {ndvi_diag["tiles"]}')
    print(f'Median gap between observations: {ndvi_diag["median_gap_days"]} days')
    print(ndvi_inventory[['date', 'tile', 'path']].tail(12).to_string(index=False))
else:
    print('No MOD13A2 HDF files were found in the configured locations.')

ndvi_blockers = []
if earthdata_root is None:
    ndvi_blockers.append('No Earthdata folder detected for MOD13A2.')
if ndvi_diag['unique_dates'] < MIN_NDVI_FILES:
    ndvi_blockers.append(
        f'Only {ndvi_diag["unique_dates"]} unique MOD13A2 composite dates are present. Re-download a longer window before trusting the overlay.'
    )
if pd.notna(ndvi_diag['median_gap_days']) and ndvi_diag['median_gap_days'] > 45:
    ndvi_blockers.append(
        f'Median MOD13A2 observation gap is {ndvi_diag["median_gap_days"]:.0f} days, which does not look like a normal 16-day vegetation series.'
    )
try:
    require_pyhdf()
except ImportError as exc:
    ndvi_blockers.append(str(exc))

if ndvi_blockers:
    print('\nNDVI blockers:')
    for item in ndvi_blockers:
        print(f'- {item}')
else:
    print('\nNDVI inventory looks sufficient to attempt parsing.')


## Corn Baseline and NDVI Regime Overlay

This section loads the local corn cache, evaluates the three course trend rules, and then applies the MODIS NDVI regime overlay to the primary `30/100` trend strategy.


In [ ]:
start_date = '2000-02-18'
corn = load_or_download_futures_cache('ZC=F', CORN_CACHE_PATH, start_date, refresh=REFRESH_CORN_FROM_YAHOO)

baseline_summaries = [summarize_strategy('buy_and_hold', corn['futures_ret'])]
for short_window, long_window in MA_PAIRS:
    strat_col = f'strategy_ret_{short_window}_{long_window}'
    signal_col = f'signal_{short_window}_{long_window}'
    baseline_summaries.append(
        summarize_strategy(
            f'ma_{short_window}_{long_window}',
            corn[strat_col],
            corn[signal_col],
        )
    )
baseline_summary = pd.DataFrame(baseline_summaries).set_index('strategy').round(4)
print(baseline_summary)

if ndvi_blockers:
    ndvi_level = pd.DataFrame(columns=['ndvi', 'ndvi_roll_mean', 'ndvi_roll_std', 'ndvi_z', 'ndvi_regime'])
    ndvi_daily = ndvi_level.copy()
    ndvi_overlay_analysis = pd.DataFrame()
    ndvi_overlay_summary = pd.DataFrame([{'status': 'blocked', 'detail': ' | '.join(ndvi_blockers)}])
    print('Skipping NDVI overlay because the current dataset is not implementation-ready.')
    print(ndvi_overlay_summary.to_string(index=False))
else:
    ndvi_level, ndvi_daily = load_ndvi_series(ndvi_inventory)
    ndvi_overlay_analysis = corn.join(ndvi_daily[['ndvi', 'ndvi_z', 'ndvi_regime']], how='inner')
    ndvi_overlay_analysis = apply_ndvi_overlay(ndvi_overlay_analysis, PRIMARY_PAIR)
    ndvi_overlay_analysis.to_csv(PROCESSED_DIR / 'corn_ndvi_overlay_panel.csv', index_label='Date')
    ndvi_overlay_summary = pd.DataFrame([
        summarize_strategy('buy_and_hold', ndvi_overlay_analysis['futures_ret']),
        summarize_strategy('ma_30_100', ndvi_overlay_analysis['strategy_ret_30_100'], ndvi_overlay_analysis['signal_30_100']),
        summarize_strategy('ma_30_100_plus_ndvi', ndvi_overlay_analysis['overlay_ret'], ndvi_overlay_analysis['overlay_signal']),
    ]).set_index('strategy').round(4)
    print('NDVI overlay summary')
    print(ndvi_overlay_summary)

fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)
((1 + corn['futures_ret'].fillna(0)).cumprod() - 1).plot(ax=axes[0], label='Buy & hold', linewidth=2)
for short_window, long_window in MA_PAIRS:
    strat_col = f'strategy_ret_{short_window}_{long_window}'
    label = f'MA {short_window}/{long_window}'
    ((1 + corn[strat_col].fillna(0)).cumprod() - 1).plot(ax=axes[0], label=label, alpha=0.85)
axes[0].set_title('Corn futures: buy-and-hold vs moving-average rules')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

if not ndvi_level.empty:
    ndvi_level['ndvi_z'].plot(ax=axes[1], color='darkorange', linewidth=1.5)
    axes[1].axhline(-1.0, color='gray', linestyle='--', linewidth=1)
    axes[1].axhline(1.0, color='gray', linestyle='--', linewidth=1)
    axes[1].set_title('MOD13A2 NDVI z-score regime')
else:
    roll_view = corn.loc[corn['is_roll_switch'], ['Close', 'Adj_Close_Futures']].copy()
    if not roll_view.empty:
        roll_view.plot(ax=axes[1], marker='o')
    axes[1].set_title('Roll-switch checkpoints in raw vs adjusted corn prices')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Additional Corn Strategies from the Course

This section ports two more course-style ideas into the agricultural setting:

- a counter-trend bounce setup based on an average-range pullback from a prior high
- a volatility-regime rule adapted from the volatility assignment


In [ ]:
countertrend = run_countertrend_strategy(corn, p=2.2)
vol_regime = run_vol_regime_strategy(corn)

corn_extra_summary = pd.DataFrame([
    summarize_strategy('countertrend_p2_2', countertrend['strategy_return'], countertrend['signal']),
    summarize_strategy('vol_regime_shifted', vol_regime['strategy_return'], vol_regime['signal']),
]).set_index('strategy').round(4)
print(corn_extra_summary)

fig, ax = plt.subplots(figsize=(12, 6))
((1 + corn['strategy_ret_10_30'].fillna(0)).cumprod() - 1).plot(ax=ax, label='MA 10/30', linewidth=2)
if not ndvi_overlay_analysis.empty:
    ((1 + ndvi_overlay_analysis['overlay_ret'].fillna(0)).cumprod() - 1).plot(ax=ax, label='MA 30/100 + NDVI', linewidth=2)
((1 + countertrend['strategy_return'].fillna(0)).cumprod() - 1).plot(ax=ax, label='Counter-trend', linewidth=2)
((1 + vol_regime['strategy_return'].fillna(0)).cumprod() - 1).plot(ax=ax, label='Vol regime', linewidth=2)
ax.set_title('Corn strategy extensions')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


## Optional Wheat and Corn-Wheat Pairs Study

If a local wheat cache exists, this section loads it and tests whether corn and wheat form a spread that looks stationary enough to justify a course-style pairs strategy.


In [ ]:
wheat = load_local_futures_cache(WHEAT_CACHE_PATH, 'wheat') if RUN_WHEAT_PAIRS_ANALYSIS else None
pairs_panel = pd.DataFrame()
pairs_strategy_summary = pd.DataFrame()
pairs_diagnostics = None

if wheat is None:
    print('Pairs analysis skipped because the wheat cache is missing. Run wheat_futures_data_builder.ipynb first.')
else:
    print(f'Wheat cache range: {wheat.index.min().date()} to {wheat.index.max().date()}')
    pairs_panel = build_pairs_panel(corn, wheat)
    pairs_panel.to_csv(PAIRS_CACHE_PATH, index_label='Date')
    pairs_diagnostics = dickey_fuller_diagnostics(pairs_panel['spread_price'])
    print('Corn-wheat spread stationarity diagnostics')
    print(pairs_diagnostics)

    pair_results = {}
    for horizon in [5, 10, 20]:
        result = run_pairs_position_strategy(pairs_panel[f'zdiff{horizon}'], pairs_panel['future_spread_ret'])
        pairs_panel[f'pairs_ret_{horizon}'] = result['strategy_return']
        pairs_panel[f'pairs_pos_{horizon}'] = result['position']
        pair_results[horizon] = result
    pairs_strategy_summary = pd.DataFrame([
        summarize_strategy('pairs_5d', pairs_panel['pairs_ret_5'], pairs_panel['pairs_pos_5']),
        summarize_strategy('pairs_10d', pairs_panel['pairs_ret_10'], pairs_panel['pairs_pos_10']),
        summarize_strategy('pairs_20d', pairs_panel['pairs_ret_20'], pairs_panel['pairs_pos_20']),
    ]).set_index('strategy').round(4)
    print('Pairs strategy summary')
    print(pairs_strategy_summary)

    fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)
    pairs_panel['spread_price'].plot(ax=axes[0], color='slateblue', linewidth=1.5)
    axes[0].set_title('Corn minus wheat spread (adjusted close)')
    axes[0].grid(True, alpha=0.3)
    ((1 + pairs_panel['pairs_ret_5'].fillna(0)).cumprod() - 1).plot(ax=axes[1], label='Pairs 5d', linewidth=2)
    ((1 + pairs_panel['pairs_ret_10'].fillna(0)).cumprod() - 1).plot(ax=axes[1], label='Pairs 10d', linewidth=2)
    ((1 + pairs_panel['pairs_ret_20'].fillna(0)).cumprod() - 1).plot(ax=axes[1], label='Pairs 20d', linewidth=2)
    axes[1].set_title('Corn-wheat pairs strategy cumulative returns')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    plt.tight_layout()
    plt.show()


## Combine Strategy Return Streams

This section treats each strategy as a sleeve, computes annualized Sharpe ratios with no beta subtraction, and then performs a coarse weight search in the spirit of the hedge-weight assignment.


In [ ]:
strategy_returns = pd.DataFrame(index=corn.index)
strategy_returns['ma_10_30'] = corn['strategy_ret_10_30']
strategy_returns['ma_30_100'] = corn['strategy_ret_30_100']
strategy_returns['ma_80_160'] = corn['strategy_ret_80_160']
strategy_returns['countertrend_p2_2'] = countertrend['strategy_return']
strategy_returns['vol_regime_shifted'] = vol_regime['strategy_return']

if not ndvi_overlay_analysis.empty:
    strategy_returns['ma_30_100_plus_ndvi'] = ndvi_overlay_analysis['overlay_ret']
if not pairs_panel.empty:
    strategy_returns = strategy_returns.join(pairs_panel[['pairs_ret_5', 'pairs_ret_10', 'pairs_ret_20']], how='left')

strategy_summary_all, best_combo, best_combo_returns, combo_candidates, equal_weight_returns = optimize_strategy_combo(
    strategy_returns,
    top_n=MAX_STRATEGIES_IN_COMBO,
    step=COMBINATION_GRID_STEP,
)
strategy_summary_all = strategy_summary_all.sort_values('ann_sharpe', ascending=False).round(4)
print('All strategy return streams ranked by Sharpe')
print(strategy_summary_all)
print('Combo candidates')
print(combo_candidates)

if best_combo is None:
    print('Not enough positive-Sharpe strategy streams were available to build a combination portfolio.')
else:
    combo_summary = pd.DataFrame([
        summarize_strategy('equal_weight_combo', equal_weight_returns),
        summarize_strategy('best_weighted_combo', best_combo_returns),
    ]).set_index('strategy').round(4)
    print('Combination summary')
    print(combo_summary)
    print('Best weights')
    print(best_combo['weights'])

    plot_frame = pd.DataFrame(index=best_combo_returns.index)
    for name in combo_candidates:
        plot_frame[name] = strategy_returns[name].reindex(best_combo_returns.index)
    plot_frame['equal_weight_combo'] = equal_weight_returns
    plot_frame['best_weighted_combo'] = best_combo_returns

    fig, ax = plt.subplots(figsize=(13, 7))
    for name in combo_candidates:
        ((1 + plot_frame[name].fillna(0)).cumprod() - 1).plot(ax=ax, linewidth=1.2, alpha=0.7, label=name)
    ((1 + plot_frame['equal_weight_combo'].fillna(0)).cumprod() - 1).plot(ax=ax, linewidth=2.5, label='equal_weight_combo')
    ((1 + plot_frame['best_weighted_combo'].fillna(0)).cumprod() - 1).plot(ax=ax, linewidth=2.8, label='best_weighted_combo')
    ax.set_title('Strategy sleeves and combined portfolios')
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=2)
    plt.tight_layout()
    plt.show()


## Notes

- The main notebook is now deliberately CSV-first. Corn can refresh from Yahoo with a flag, but wheat is expected to be prepared through the dedicated builder notebook.
- The Sharpe ratio used throughout is the simple annualized mean-over-volatility version requested for this project, with no beta or benchmark subtraction.
- The pairs section is empirical rather than assumed: if the corn-wheat spread does not look stationary enough, treat those results as exploratory rather than as a production-quality edge.
- The weighted-combination section is a coarse search, not a rigorous optimizer. It is best used to surface promising sleeves for follow-up testing.
